# Exercise 1 - operational vs analytical


In [ ]:
from faker.providers import BaseProvider
from faker import Faker
from dataclasses import dataclass
from datetime import datetime

fake = Faker()
fake.seed_instance(0)
name_pool = [fake.name() for _ in range(5)]


@dataclass
class Transaction:
    id: int
    name: str
    datetime: datetime
    amount: float


class TransactionProvider(BaseProvider):
    __provider__ = "transaction"

    def transaction(self) -> Transaction:
        return Transaction(
            fake.unique.uuid4(),
            fake.random_element(elements=name_pool),
            fake.date_time_this_century(),
            fake.pyfloat(min_value=-120, max_value=100),
        )


fake.add_provider(TransactionProvider)
N = int(500)
rowbased = [fake.transaction() for _ in range(N)]

In [ ]:
rowbased

In [ ]:
columnar = {
    "id": [t.id for t in rowbased],
    "name": [t.name for t in rowbased],
    "datetime": [t.datetime for t in rowbased],
    "amount": [t.amount for t in rowbased],
}

Calculate the maximal transaction amount in the row-based approach.


In [ ]:
%%timeit

max(rowbased, key=lambda t: t.amount)

In [ ]:
%%timeit

max([row.amount for row in rowbased])

In [ ]:
%%timeit

max_amount = float("-inf")
for row in rowbased:
    if row.amount > max_amount:
        max_amount = row.amount
max_amount

Calculate the sum of all transactions by Jorge Sullivan in the row-based approach.


In [ ]:
%%timeit

sum([row.amount for row in rowbased if row.name == "Jorge Sullivan"])

In [ ]:
%%timeit

sum_value = 0
for t in rowbased:
    sum_value += t.amount
sum_value

Calculate the maximum transaction amount in the columnar approach


In [ ]:
%%timeit

max(columnar["amount"])

Calculate the sum of all transactions by Jorge Sullivan in the columnar approach.


In [ ]:
%%timeit

# This is slower than row-based in pure python because :
# - zip adds overhead
# - still iterating over rows, negating the benefit of columnar storage
# The advantages of columnar storage for analytical queries become more apparent when using vectorized operations (numpy, pandas, polars, etc.)
sum(amount for amount, name in zip(columnar["amount"], columnar["name"]) if name == "Jorge Sullivan")

Retrieve all information for the transaction with ID `989bb030-86c4-4b06-86a8-e22fd57caf0d`, both in the rowbased and in the columnar approach.


In [ ]:
%%timeit

next(filter(lambda t: t.id == "989bb030-86c4-4b06-86a8-e22fd57caf0d", rowbased))

In [ ]:
%%timeit

[t for t in rowbased if t.id == "989bb030-86c4-4b06-86a8-e22fd57caf0d"][0]

In [ ]:
%%timeit

# Also this is a contraditcion in pure python against actual database engines.
# Transactional databases are optimized with indexes to retreive rows by those indexed column values.
# Here in pure python we do not have that, but we can use the list.index() mehtod which seems to be actually faster
# than pure python loops in the rowbased approach.

idx = [columnar["id"].index("989bb030-86c4-4b06-86a8-e22fd57caf0d")][0]

Transaction(
    id=columnar["id"][idx],
    name=columnar["name"][idx],
    datetime=columnar["datetime"][idx],
    amount=columnar["amount"][idx],
)